# Reference-guided object search

**Recipe 82** · Level 3 (Intermediate) · Decision types: `Box` + `Noul`

Sources:

- S02: [TypeSafe AI: Primitives (Questions)](https://docs.typesafe.ai/primitives)
- S03: [TypeSafe AI: Confidence](https://docs.typesafe.ai/confidence)
- S09: [TypeSafe AI: Images](https://docs.typesafe.ai/images)
- S10: [TypeSafe AI: Box](https://docs.typesafe.ai/primitives/box)
- S11: [TypeSafe AI: Models and long context](https://docs.typesafe.ai/models)
- S12: [TypeSafe AI: Jev 1.14 release](https://typesafe.ai/blog/jev-1-14)


## What you will build

Find a fictional product on a cluttered shelf from its reference image. The reference is embedded directly in the `Box` instructions, so the identity comes from the supplied image. Python first requires two bounded judgments: the product is present, and exactly one instance is visible. Only then does it request a rectangle and simulate a highlight.

All images here are authored **synthetic drawings**, including the reference portraits; they stand in for a product-photo workflow and are not photographs or model-generated observations. The lesson compares this staged search with a real pixel-template search. No robot moves and no product is picked. Images and `Box` use experimental API capabilities ([S09](https://docs.typesafe.ai/images), [S10](https://docs.typesafe.ai/primitives/box)); the live path pins `jev-1.14.0` ([S11](https://docs.typesafe.ai/models)).

## Setup and run mode

Run from this folder. The fixtures contain 26 independently drawn shelf scenes: 12 validation examples, 12 held-out test examples and two unscored demonstrations. Three product reference images are shared across splits, as a known catalog would be; no query image or scene layout is reused. Pillow and NumPy handle the images, and the cookbook handles typed answers and replay. A [replay key](../../docs/glossary.md#replay-key) is the identity of a request; it includes the bytes of both images here.

The header identifies the [run mode](../../docs/glossary.md#run-mode), meaning whether answers are synthetic, recorded or live. Synthetic results only check this authored pipeline. Direct live execution is unrecorded exploration; capture answers first for reportable held-out results.

In [ ]:
import json
from pathlib import Path

from IPython.display import display
from matplotlib.figure import Figure
from PIL import Image

from jev_cookbook import get_backend, load_helpers
from jev_cookbook.fixtures import load_inputs, load_labels, responses_path
from jev_cookbook.simulation import ActionLog, ReviewQueue
from jev_cookbook.style import apply_style, run_header, show_answer
from jev_cookbook.vision import draw_boxes, normalized_to_pixels, run_record

apply_style()
helpers = load_helpers()
examples = load_inputs()
labels = load_labels()
backend = get_backend(fixtures=responses_path(), images=True, model="jev-1.14.0")
scored = [e for e in examples if e.split in ("validation", "test")]
offline = backend.mode in ("synthetic", "scripted")
check = (
    " (a pipeline check, not a Jev result)"
    if offline
    else (" (unrecorded exploration; not a reportable result)" if backend.mode == "live" else "")
)
selection = " (validation selection step)"
run_header(82, "Reference-guided object search", backend=backend, n_examples=len(scored))

## The state

The state contains only the query image reference. The reference product appears in question instructions, with both assets supplied to the backend. Local filenames, split membership, scene construction parameters and annotations stay in Python. Content-addressed image references carry no product name or expected answer.

First inspect the independent reference portrait, then the cluttered shelf. Similar colors are insufficient: the cap, body and emblem together define the product. This drawing has no real manufacturer or commercial identity.

In [ ]:
demo = [e for e in examples if e.split == "demo"]
example = demo[0]
state, questions, assets = helpers.request(example.fields)
print("State:", state)
print("Reference portrait, followed by shelf scene:")
with Image.open(example.fields["reference"]) as image:
    display(image.copy())
with Image.open(example.fields["query"]) as image:
    display(image.copy())

## The questions

`Noul` supplies a probability that one proposition is true ([S02](https://docs.typesafe.ai/primitives)). The two independent propositions, “at least one matching instance is visible” and “exactly one matching instance is visible,” share one request. They are not object-count arithmetic: Python only needs these bounded judgments.

Before seeing validation answers, we fix acquisition rules: both probabilities at least 0.8 permit one `Box` request; both at most 0.2 resolve absence; every other combination goes to [review](../../docs/glossary.md#review), an explicit deferral for a person. In particular, high presence with low uniqueness never authorizes choosing an arbitrary instance. The contradictory low-presence/high-uniqueness combination also defers.

A `Box` always returns a rectangle, even for an absent object ([S10](https://docs.typesafe.ai/primitives/box)). Its request is therefore dependent on screening. The actual reference image below is inside the instructions, not merely named in prose or placed only in state.

In [ ]:
for name, question in questions.items():
    print(name, question.to_dict())
_, box_questions, _ = helpers.request(example.fields, stage="box")
print("Dependent localization:", box_questions["target"].to_dict())

## One answer up close

The [provenance](../../docs/glossary.md#provenance) attached to an answer tells us where it came from. These probabilities and rectangles were authored for replay. A Noul probability is not an extra confidence field. Box confidence may be missing; Python treats that as unknown.

Coordinates are normalized to the query image: left/right divide by width, and top/bottom divide by height. The display converts them to pixels and draws the returned rectangle before any aggregate result is computed.

In [ ]:
evidence = {e.id: helpers.search(backend, e.fields) for e in demo}
close_up = evidence[example.id]
for answer in close_up.screen.answers.values():
    show_answer(answer)
if close_up.located is not None:
    box_answer = close_up.located.answers["target"]
    show_answer(box_answer)
    with Image.open(example.fields["query"]) as image:
        print("Pixel bounds:", normalized_to_pixels(box_answer, *image.size))
        display(draw_boxes(image, {"returned product": box_answer}))
print("Duplicate-match demonstration screening:", helpers.screening(evidence[demo[1].id].screen))

## Python's part

A [confidence gate](../../docs/glossary.md#confidence-gate) is a rule that accepts a prediction only above a chosen threshold. Validation will select the Box gate; the separate presence and uniqueness acquisition rules stay fixed. The recipe allows at most two requests per scene, has no retry loop and never asks a second localization question. The live backend also caps HTTP attempts, including its retries.

Python owns geometry, the request budget, the accepted/absent/review outcomes, and the simulated `ActionLog` and `ReviewQueue`. Missing Box confidence, exhausted budget or failed screening means review, even if some other rectangle is supplied. A final absence records no highlight. Reviews never count as answered examples.

The code below demonstrates a one-request budget stop using already-cached evidence. Its zero threshold cannot override the missing dependent stage.

In [ ]:
budget_stop = helpers.Evidence(close_up.screen, None, 1)
print("One-request budget outcome:", helpers.route(budget_stop, threshold=0.0))
print("Duplicate scene outcome:", helpers.route(evidence[demo[1].id], threshold=0.0))
print("Acquisition rules:", helpers.PRESENCE_FLOOR, helpers.ABSENCE_CEILING)

## Evaluation

A [gold label](../../docs/glossary.md#gold-label) is the authored answer used to score an example. Here it is a readable list of matching product rectangles: empty means absent, one means uniquely localizable, and two means ambiguous. For partly occluded bottles, the outer visible extremities still determine the box. Intersection over union (IoU) is the area shared by two boxes divided by their combined area; a unique localization is correct at IoU at least 0.5. An accepted absence is correct only for an empty list. A multi-instance scene must defer, and accepting any single instance is an error.

[Coverage](../../docs/glossary.md#coverage) is the fraction of scenes answered rather than deferred. [Accuracy](../../docs/glossary.md#accuracy) here is the fraction of accepted answers that are correct. `evaluate_outcomes` uses the actual route decisions, including unconditional reviews. Localization metrics apply only to resolved zero/one-target scenes; reviewed scenes and multi-instance cases have separate counts. A resolved false absence contributes a miss and zero IoU. A box on an absent target contributes a false positive. A wrong box on a present target contributes low IoU, not an invented absence. A second, whole-split delivery measure averages delivered-box IoU over every uniquely present target, with zero when no box is delivered, including review. Its undelivered count includes both deferrals and false absences; it is not a presence-miss count. This exposes the cost of abstention without treating review as a prediction of absence.

### Validation selects, then freezes

The Box candidate family is the distinct known confidences among acquired validation boxes. Select the lowest threshold achieving `target_accuracy=1.0` on that eligible pool, maximizing its accepted size. This empirical selection objective is not a future accuracy guarantee. Unknown confidence and uncertain/inconsistent screening are excluded from selection and remain review outcomes. Confident absence is also excluded from Box selection, but it is a gated, accepted final answer in full-pipeline accounting.

The pixel baseline extracts foreground from the reference against its corner background, slides RGB templates over quarter-resolution scene pixels at scales 0.45, 0.50 and 0.60, and suppresses overlapping matches. It never reads labels or the drawing specification. It searches upright templates only, so rotation and occlusion are expected failure modes. Multiple sufficiently similar templates defer. Choose its error tolerance from `[0.06, 0.10, 0.16, 0.22]` to minimize `3 × accepted errors + reviews`, subject to at least 50% coverage; ties prefer the smaller tolerance. Both objectives and candidate families are declared before looking at test outcomes.

In [ ]:
validation = [e for e in examples if e.split == "validation"]
evidence.update({e.id: helpers.search(backend, e.fields) for e in validation})
threshold = helpers.choose_gate(validation, evidence, labels)
validation_outcomes = {e.id: helpers.route(evidence[e.id], threshold) for e in validation}
validation_report = helpers.summarize(validation, validation_outcomes, labels)
print(
    f"Box gate: {threshold:.2f}; eligible N={len(helpers.validation_pairs(validation, evidence, labels))}{selection}{check}"
)
print(f"Full validation N={len(validation)}: {validation_report}{selection}{check}")

baseline_candidates = {
    e.id: helpers.template_candidates(Path(e.fields["query"]), Path(e.fields["reference"]))
    for e in validation
}
tolerance, baseline_sweep = helpers.choose_template_tolerance(
    validation, baseline_candidates, labels
)
for row in baseline_sweep:
    print(f"Pixel tolerance candidate N={len(validation)}: {row}{selection}{check}")
frozen = (threshold, tolerance)
print(f"Frozen Box gate and pixel tolerance: {frozen}; N={len(validation)}{selection}{check}")

### Held-out comparison

The following cell reports test outcomes once using the frozen settings. Both methods see the same test images and reference portraits. The numbers beside the chart are the values plotted. Separate denominators prevent review from being relabeled as absence or quietly counted as successful localization.

In [ ]:
test = [e for e in examples if e.split == "test"]
threshold, tolerance = frozen
evidence.update({e.id: helpers.search(backend, e.fields) for e in test})
test_outcomes = {e.id: helpers.route(evidence[e.id], threshold) for e in test}
baseline_candidates.update(
    {
        e.id: helpers.template_candidates(Path(e.fields["query"]), Path(e.fields["reference"]))
        for e in test
    }
)
baseline_outcomes = {
    e.id: helpers.template_outcome(baseline_candidates[e.id], tolerance) for e in test
}
reports = {
    "Staged reference search": helpers.summarize(test, test_outcomes, labels),
    "Pixel templates": helpers.summarize(test, baseline_outcomes, labels),
}
for name, report in reports.items():
    print(f"{name}; test N={len(test)}: {report}{check}")

fig = Figure(figsize=(8, 3.6))
ax = fig.subplots()
for offset, (metric, color) in enumerate(
    (("coverage", "#1E1E1E"), ("accepted_accuracy", "#F386A1"))
):
    values = [report[metric] for report in reports.values()]
    print(f"Plotted {metric}; test N={len(test)}: {values}{check}")
    ax.bar(
        [index + offset * 0.35 for index in range(len(reports))],
        values,
        width=0.35,
        label=metric,
        color=color,
    )
ax.set_xticks([0.175, 1.175], list(reports))
ax.set_ylim(0, 1.1)
ax.set_ylabel("Fraction")
ax.legend(loc="lower right")
ax.set_title(f"Test N={len(test)}{check}", fontsize=10)
fig.tight_layout()
fig

### Inspect the accepted mistakes

The authored test responses include a confident false detection on a lookalike-only shelf, a box on the wrong product, and a confident false absence under dim lighting. These errors survive the frozen policy. A correct validation subset cannot establish reliable presence or product identity on new images.

The next output lists accepted errors and renders the first one from the already-computed outcomes. The action and review records are simulations: `executed` remains false, and nothing is picked or sent.

In [ ]:
wrong = [
    e
    for e in test
    if test_outcomes[e.id].outcome != helpers.REVIEW
    and not helpers.correct(test_outcomes[e.id], labels[e.id])
]
print(f"Accepted wrong examples; test N={len(test)}: {[e.id for e in wrong]}{check}")
for e in wrong:
    print(e.id, labels[e.id]["case"], test_outcomes[e.id])
if wrong:
    e = wrong[0]
    with Image.open(e.fields["query"]) as image:
        if test_outcomes[e.id].box is not None:
            display(draw_boxes(image, {"accepted wrong box": test_outcomes[e.id].box}))

actions, reviews = ActionLog(), ReviewQueue()
for e in test:
    helpers.simulate(e.id, test_outcomes[e.id], actions, reviews)
print("Simulated ActionLog:", json.dumps(actions.to_dicts(), indent=2))
print("Pending ReviewQueue:", json.dumps(reviews.to_dicts(), indent=2))

## What was and was not measured

The cell below prints the run mode, N for each scored split, and the actual request count. Recorded/live answers additionally expose the returned model, capture dates and reported usage. Model name, tokens, inference latency and cost are unknown for synthetic replay and stay `None`. Replay wall time is never inference latency. This notebook does not measure billing or inference latency.

There are 52 stored responses, two per scene. Counterfactual Box answers, including absent targets, are kept to test the API contract and fixture identity. The notebook requests only the dependent branches it can use, once per scene, and reuses those results. The upper bound is 52 logical requests; a live retry consumes an additional backend attempt. Direct live exploration is not a reportable held-out measurement. See the README for explicit recording with the same state/question/asset triples.

In [ ]:
results = [
    result for ev in evidence.values() for result in (ev.screen, ev.located) if result is not None
]
accounting = [run_record(result, mode=backend.mode) for result in results]
actual_requests = sum(ev.requests for ev in evidence.values())
if offline:
    print("Not measured live: a pipeline check, not a Jev result.")
    print(
        f"N: validation={len(validation)}, test={len(test)}, demo={len(demo)}; mode={backend.mode}"
    )
else:
    print(
        f"N: validation={len(validation)}, test={len(test)}, demo={len(demo)}; mode={backend.mode}"
    )
    dates = sorted(
        {answer.provenance.date for result in results for answer in result.answers.values()}
    )
    print("Returned models:", sorted({result.model for result in results}), "capture dates:", dates)
    if backend.mode == "live":
        print("Unrecorded exploration: no reportable held-out result.")
print(
    f"Actual requests={actual_requests}; stored stage responses=52; logical upper bound=52; N={len(examples)}{check}"
)
print("Accounting example:", accounting[0])
print("Latency seconds:", None, "Cost USD:", None)
print("Per-call token usage:", [(row["input_tokens"], row["output_tokens"]) for row in accounting])

## Next steps

Replace the drawings with independently captured, licensed product references and shelf photos. Write visible-extent annotations before recording, add more product families and camera conditions, and keep a fresh holdout when prompts or thresholds change. The simple pixel baseline assumes a plain reference background and searches only a small scale range; a stronger baseline could add rotated templates before a new validation/test cycle.

Continue with [abstention and review queues](../38-abstention-and-review-queues/) to study deferral policies, or [sentiment classification](../01-sentiment-classification/) for selection discipline. The [vision helpers](../../docs/vision.md) explain normalized coordinates, image identity and localization denominators. A human must resolve multiple matching instances or choose a new reference view; a rectangle alone cannot establish identity.